# Data Monitoring — Metrics Analysis

Visualize observation data exported from the data-monitoring-service.

## Setup

Export the metrics from the SPARQL endpoint using the query below, save as `metrics.csv`:

```sparql
PREFIX mon: <http://lblod.data.gift/vocabularies/monitoring/>
PREFIX dct: <http://purl.org/dc/terms/>

SELECT ?queryName ?metricName ?value ?created
WHERE {
  ?obs a mon:Observation ;
    mon:forQuery ?query ;
    mon:metricName ?metricName ;
    mon:value ?value ;
    mon:partOfRun ?run .
  ?run dct:created ?created .
  BIND(REPLACE(STR(?query), "^.*/", "") AS ?queryName)
}
ORDER BY ?queryName DESC(?created) ?metricName
```

Expected CSV columns: `queryName`, `metricName`, `value`, `created`

In [ ]:
%pip install -q pandas matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

CSV_PATH = "metrics.csv"

df = pd.read_csv(CSV_PATH)
df["created"] = pd.to_datetime(df["created"])
df["value"] = pd.to_numeric(df["value"])
df.sort_values("created", inplace=True)

print(f"{len(df)} observations, {df['queryName'].nunique()} queries, {df['created'].nunique()} runs")
df.head()

## Latest snapshot

Most recent values per query and metric.

In [ ]:
latest = df.loc[df.groupby(["queryName", "metricName"])["created"].idxmax()]

# Pivot: rows = queries, columns = metrics
pivot = latest.pivot(index="queryName", columns="metricName", values="value")

# Reorder so totalRows comes first
cols = ["totalRows"] + sorted(c for c in pivot.columns if c != "totalRows")
pivot = pivot[[c for c in cols if c in pivot.columns]]

pivot

## Coverage

For each query, what percentage of rows have the optional values filled in?

In [ ]:
completeness_cols = [c for c in pivot.columns if c.startswith("completeness_")]
if completeness_cols and "totalRows" in pivot.columns:
    coverage = pivot[completeness_cols].div(pivot["totalRows"], axis=0) * 100
    coverage = coverage.round(1)

    fig, ax = plt.subplots(figsize=(14, max(4, len(coverage) * 0.4)))
    coverage.plot.barh(ax=ax)
    ax.set_xlabel("Coverage (%)")
    ax.set_title("Optional field coverage by query")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.set_xlim(0, 105)
    plt.tight_layout()
    plt.show()
else:
    print("No completeness metrics found")

## Row count over time

Track `totalRows` per query across runs to spot sudden drops or spikes.

In [ ]:
totals = df[df["metricName"] == "totalRows"]

if len(totals) and totals["created"].nunique() > 1:
    fig, ax = plt.subplots(figsize=(14, 6))
    for name, group in totals.groupby("queryName"):
        ax.plot(group["created"], group["value"], marker="o", markersize=3, label=name)
    ax.set_ylabel("totalRows")
    ax.set_title("Row count over time")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 runs to plot trends")

## Coverage over time

Track how optional field coverage evolves across runs for a specific query.

In [ ]:
# Pick the query with the most completeness metrics to show
queries_with_completeness = (
    df[df["metricName"].str.startswith("completeness_")]
    .groupby("queryName")["metricName"]
    .nunique()
    .sort_values(ascending=False)
)
QUERY = queries_with_completeness.index[0] if len(queries_with_completeness) else None

if QUERY and df["created"].nunique() > 1:
    qdf = df[df["queryName"] == QUERY].copy()
    totals_q = qdf[qdf["metricName"] == "totalRows"].set_index("created")["value"]
    comp_q = qdf[qdf["metricName"].str.startswith("completeness_")]

    fig, ax = plt.subplots(figsize=(14, 5))
    for metric, group in comp_q.groupby("metricName"):
        merged = group.set_index("created").join(totals_q.rename("total"))
        pct = (merged["value"] / merged["total"] * 100).dropna()
        ax.plot(pct.index, pct.values, marker="o", markersize=3, label=metric.replace("completeness_", ""))
    ax.set_ylabel("Coverage (%)")
    ax.set_ylim(-5, 105)
    ax.set_title(f"Coverage over time — {QUERY}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print(f"Need at least 2 runs with completeness metrics to plot (selected query: {QUERY})")

## Change detection

Percentage change between the latest and previous run per metric. Highlights potential anomalies.

In [ ]:
runs = sorted(df["created"].unique())
if len(runs) >= 2:
    prev = df[df["created"] == runs[-2]].set_index(["queryName", "metricName"])["value"]
    curr = df[df["created"] == runs[-1]].set_index(["queryName", "metricName"])["value"]
    combined = pd.DataFrame({"previous": prev, "current": curr}).dropna()
    combined["change_%"] = ((combined["current"] - combined["previous"]) / combined["previous"] * 100).round(2)
    combined = combined.sort_values("change_%")

    flagged = combined[combined["change_%"].abs() > 10]
    if len(flagged):
        print(f"{len(flagged)} metrics changed by more than 10%:\n")
        display(flagged.style.applymap(
            lambda v: "background-color: #ffcccc" if isinstance(v, (int, float)) and abs(v) > 25
            else ("background-color: #fff3cd" if isinstance(v, (int, float)) and abs(v) > 10 else ""),
            subset=["change_%"]
        ))
    else:
        print("No significant changes between the last two runs")
else:
    print("Need at least 2 runs to compare")